## TimesFM EEG evaluation based on Parquet from `parquet_maker.py`

parquet data, 4 channels, all subjects, 5 separate windows(ctx=512, h=64)

In [ ]:
!pip install "timesfm[torch]" --ignore-requires-python -q
print("Restart kernel before running script further")

## Config

In [1]:
import os
import numpy as np
import pandas as pd
import torch  
import timesfm
from sklearn.metrics import mean_squared_error

PARQUET_PATH = os.environ.get(
    "EEG_PARQUET", os.path.join(os.getcwd(), "eeg_preprocessed_4ch_raw.parquet")
)
KAGGLE_DATASET_DIR = "/kaggle/input/datasets/michalzienkowicz/pp-eeg-4ch-raw"
PARQUET_FILENAME = "eeg_preprocessed_4ch_raw.parquet"
DATA_PATH = os.path.join(KAGGLE_DATASET_DIR, PARQUET_FILENAME)
PARQUET_PATH = DATA_PATH
CSV_OUT = "timesfm_eeg_cov_eval_results.csv"

LIMIT_PATIENTS = None
CHANNELS = ["Fp1", "Fp2", "P3", "P4"]
CONTEXT_LENGTH = 512
HORIZON_LENGTH = 64
N_WINDOWS = 5
FS = 500 # Native sampling rate
OFFSET_SAMPLES = 3 * FS

COVARIATE_MAP = {
    "Fp1": "P3",
    "P3": "Fp1",
    "Fp2": "P4",
    "P4": "Fp2"
}

df = pd.read_parquet(PARQUET_PATH)
df_eval = df.head(LIMIT_PATIENTS).copy() if LIMIT_PATIENTS else df.copy()
n_total = len(df)
print(
    f"Loaded parquet: {PARQUET_PATH} | rows in file: {n_total}, "
    f"evaluation on (the first) {len(df_eval)} subjects."
)

 See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.
Loaded PyTorch TimesFM, likely because python version is 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0].
Loaded parquet: /kaggle/input/datasets/michalzienkowicz/pp-eeg-4ch-raw/eeg_preprocessed_4ch_raw.parquet | rows in file: 88, evaluation on (the first) 88 subjects.


## Model

In [2]:
repo_id = "google/timesfm-1.0-200m-pytorch"
device = "gpu" if torch.cuda.is_available() else "cpu"

print(f"Loading TimesFM ({repo_id}), device: {device}...")

# API TimesFM
tfm = timesfm.TimesFm(
    hparams=timesfm.TimesFmHparams(
        backend=device,
        context_len=CONTEXT_LENGTH,
        horizon_len=HORIZON_LENGTH,
        per_core_batch_size=32,
    ),
    checkpoint=timesfm.TimesFmCheckpoint(
        huggingface_repo_id=repo_id
    )
)

print("TimesFM ready.")

Loading TimesFM (google/timesfm-1.0-200m-pytorch), device: gpu...


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

TimesFM ready.


## Processing functions

In [3]:
def window_starts(total_len: int, ctx: int, hor: int, n_win: int, offset: int = OFFSET_SAMPLES) -> np.ndarray:
    need = ctx + hor
    available_len = total_len - offset
    if available_len < need:
        raise ValueError(f"Signal too short: available={available_len}, required >= {need}")
    hi = total_len - need
    
    return np.linspace(offset, hi, n_win, dtype=int)


def series_to_numpy(cell) -> np.ndarray:
    if isinstance(cell, (list, np.ndarray)):
        return np.asarray(cell, dtype=np.float64)
    return np.asarray(cell.tolist() if hasattr(cell, "tolist") else cell, dtype=np.float64)


# dependent on the model used
def timesfm_batch_forecast(tfm_model, contexts: list, covariates: dict, pred_len: int) -> np.ndarray:
    
    point_forecasts, _ = tfm_model.forecast_with_covariates(
        inputs=contexts,
        dynamic_numerical_covariates=covariates
    )
    point_forecasts = np.array(point_forecasts)
    # TimesFM might return longer forecast based on output_patch_len, we trim it:
    return point_forecasts[:, :pred_len]

# Evaluation loop

In [4]:
detail_rows = []
done = 0

for _, row in df_eval.iterrows():
    sid = row["subject_id"]

    grp = row["group"] if "group" in row.index and pd.notna(row["group"]) else "Unknown"
    for ch in CHANNELS:
        cov_ch = COVARIATE_MAP[ch]
        y_target = series_to_numpy(row[ch])
        y_cov = series_to_numpy(row[cov_ch])
        starts = window_starts(len(y_target), CONTEXT_LENGTH, HORIZON_LENGTH, N_WINDOWS)

        batch_contexts = []
        batch_covariates_list = []

        for s in starts:
            # main signal context (e.g. Fp1): len 512
            batch_contexts.append(y_target[s : s + CONTEXT_LENGTH])
            
            # Covariate (e.g. P3): len 576 (512 history + 64 future)
            batch_covariates_list.append(y_cov[s : s + CONTEXT_LENGTH + HORIZON_LENGTH])
            
        # Covariate formatting for TimesFM:
        # Dict with 2D array (N_WINDOWS, 576)
        dynamic_covariates = {
            f"{cov_ch}_signal": np.array(batch_covariates_list)
        }
        
        # Predictions
        preds = timesfm_batch_forecast(tfm, batch_contexts, dynamic_covariates, HORIZON_LENGTH)
        
        if preds.shape != (N_WINDOWS, HORIZON_LENGTH):
            preds = np.asarray(preds).reshape(N_WINDOWS, HORIZON_LENGTH)
            
        mses_win = []
        for i, s in enumerate(starts):
            tgt = y_target[s + CONTEXT_LENGTH : s + CONTEXT_LENGTH + HORIZON_LENGTH]
            mses_win.append(mean_squared_error(tgt, preds[i]))
            
        mse_mean = float(np.mean(mses_win))
        detail_rows.append(
            {
                "record_type": "per_patient_electrode",
                "subject_id": sid,
                "group": grp,
                "electrode": ch,
                "covariate_used": cov_ch,
                "mse": mse_mean,
                "n_windows": N_WINDOWS,
            }
        )
    done += 1
    print(f"Subjects analysed: {done} / {len(df_eval)} (last: {sid}).")

df_detail = pd.DataFrame(detail_rows)

summary_rows = []
for g in sorted(df_detail["group"].dropna().unique()):
    sub = df_detail[(df_detail["record_type"] == "per_patient_electrode") & (df_detail["group"] == g)]
    if sub.empty:
        continue
    summary_rows.append(
        {
            "record_type": "group_all_electrodes",
            "subject_id": "",
            "group": g,
            "electrode": "ALL",
            "covariate_used": "ALL",
            "mse": float(sub["mse"].mean()),
            "n_windows": N_WINDOWS,
        }
    )
    for ch in CHANNELS:
        sub_ch = sub[sub["electrode"] == ch]
        if sub_ch.empty:
            continue
        summary_rows.append(
            {
                "record_type": "group_per_electrode",
                "subject_id": "",
                "group": g,
                "electrode": ch,
                "covariate_used": COVARIATE_MAP[ch],
                "mse": float(sub_ch["mse"].mean()),
                "n_windows": N_WINDOWS,
            }
        )

df_out = pd.concat([df_detail, pd.DataFrame(summary_rows)], ignore_index=True)
df_out.to_csv(CSV_OUT, index=False)
print(f"Saved: {CSV_OUT}")

try:
    from IPython.display import display
    display(df_out.tail(20)) # display end of csv
except ImportError:
    print(df_out.tail(20).to_string())

Subjects analysed: 1 / 88 (last: sub-001).
Subjects analysed: 2 / 88 (last: sub-002).
Subjects analysed: 3 / 88 (last: sub-003).
Subjects analysed: 4 / 88 (last: sub-004).
Subjects analysed: 5 / 88 (last: sub-005).
Subjects analysed: 6 / 88 (last: sub-006).
Subjects analysed: 7 / 88 (last: sub-007).
Subjects analysed: 8 / 88 (last: sub-008).
Subjects analysed: 9 / 88 (last: sub-009).
Subjects analysed: 10 / 88 (last: sub-010).
Subjects analysed: 11 / 88 (last: sub-011).
Subjects analysed: 12 / 88 (last: sub-012).
Subjects analysed: 13 / 88 (last: sub-013).
Subjects analysed: 14 / 88 (last: sub-014).
Subjects analysed: 15 / 88 (last: sub-015).
Subjects analysed: 16 / 88 (last: sub-016).
Subjects analysed: 17 / 88 (last: sub-017).
Subjects analysed: 18 / 88 (last: sub-018).
Subjects analysed: 19 / 88 (last: sub-019).
Subjects analysed: 20 / 88 (last: sub-020).
Subjects analysed: 21 / 88 (last: sub-021).
Subjects analysed: 22 / 88 (last: sub-022).
Subjects analysed: 23 / 88 (last: sub-023

,record_type,subject_id,group,electrode,covariate_used,mse,n_windows
347,per_patient_electrode,sub-087,F,P4,Fp2,1.088654e-10,5
348,per_patient_electrode,sub-088,F,Fp1,P3,1.982661e-10,5
349,per_patient_electrode,sub-088,F,Fp2,P4,1.645334e-10,5
350,per_patient_electrode,sub-088,F,P3,Fp1,2.034873e-10,5
351,per_patient_electrode,sub-088,F,P4,Fp2,1.500260e-10,5
352,group_all_electrodes,,A,ALL,ALL,2.143465e-10,5
353,group_per_electrode,,A,Fp1,P3,2.023233e-10,5
354,group_per_electrode,,A,Fp2,P4,2.274186e-10,5
355,group_per_electrode,,A,P3,Fp1,2.075263e-10,5
356,group_per_electrode,,A,P4,Fp2,2.201180e-10,5
